# 02. Score Quality

`papers_raw.csv`를 받아 정량 퀄리티 점수를 매깁니다.

**점수 산식 (단순 가중합)**:
- 인용수 백분위 (해당 검색 결과 안에서) × 0.30
- **연식 보정 인용수**(citations_per_year) 백분위 × 0.30  ← 2014년 100인용 vs 2024년 50인용 같은 불공정 보정
- 최신성 (FROM_YEAR → 0.0, TO_YEAR → 1.0 선형) × 0.20
- 저널/소스 보유 여부 × 0.10
- abstract 충분도 (200자 이상이면 1) × 0.10

이 점수는 **후보 좁히기를 위한 정량 신호**일 뿐입니다. 정성 평가는 `reference_quality_check(rr)` 프롬프트로 Claude에 맡기세요.

> ⚠️ `citations_per_year` 컬럼은 **노트북 01의 개선판이 만들어준 컬럼**입니다. 옛 papers_raw.csv를 쓰면 자동 계산으로 폴백합니다.


In [ ]:
!pip install -r ../../../requirements.txt

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / 'papers_raw.csv')
print(f'{len(df)} papers loaded')
df.head()

In [ ]:
FROM_YEAR = 2020   # 01에서 사용한 값
TO_YEAR = 2026

def score(df, from_year=FROM_YEAR, to_year=TO_YEAR):
    """5개 신호의 가중합. citations_per_year가 없으면 자동 계산."""
    cit = df['cited_by_count'].fillna(0).astype(float)
    cit_pct = cit.rank(pct=True) if cit.max() != cit.min() else pd.Series([0.5] * len(df), index=df.index)

    # 연식 보정 인용수 — 옛 csv에 컬럼이 없으면 자동 계산
    if 'citations_per_year' in df.columns:
        cpy = df['citations_per_year'].fillna(0).astype(float)
    else:
        year_for_age = df['year'].fillna(to_year).astype(float)
        age = (to_year - year_for_age).clip(lower=1)
        cpy = (cit / age).round(2)
    cpy_pct = cpy.rank(pct=True) if cpy.max() != cpy.min() else pd.Series([0.5] * len(df), index=df.index)

    year = df['year'].fillna(from_year).astype(float)
    recency = ((year - from_year) / max(to_year - from_year, 1)).clip(0, 1)

    has_venue = df['venue'].notna().astype(float)
    abs_ok = df['abstract'].fillna('').str.len().ge(200).astype(float)

    return (
        cit_pct  * 0.30
        + cpy_pct  * 0.30
        + recency  * 0.20
        + has_venue * 0.10
        + abs_ok    * 0.10
    ).round(3)

df['quality_score'] = score(df)
df_sorted = df.sort_values('quality_score', ascending=False).reset_index(drop=True)

# 표시 컬럼 — citations_per_year 추가 (있으면)
display_cols = ['title', 'year', 'cited_by_count']
if 'citations_per_year' in df_sorted.columns:
    display_cols.append('citations_per_year')
display_cols += ['venue', 'quality_score']
df_sorted[display_cols].head(15)


In [ ]:
out = DATA_DIR / 'papers_scored.csv'
df_sorted.to_csv(out, index=False)
print(f'Saved → {out.resolve()}')

## 다음 단계

`03_export_to_claude.ipynb`로 상위 N개를 Claude Desktop에 붙여넣을 markdown 표로 export 하세요.